In [15]:
prices = []
with open('d:\\b.txt', 'rb') as f:
    while True:
        # parsing line
        line = f.readline()
        if not line:
            break
        if line[0] == 0xe2:
            prices.append(line[3:].strip().decode())
prices.sort()            
print(prices)

['1,051', '1,251', '1,277', '1,530', '1,662', '1,663', '1,690', '1,756', '1,783', '1,845', '1,884', '1,996', '15,630', '17,423', '2,028', '2,355', '2,389', '2,408', '2,435', '2,457', '2,466', '2,509', '2,533', '2,541', '2,541', '2,638', '2,742', '3,060', '3,087', '3,167', '3,233', '3,234', '3,296', '3,313', '3,757', '3,792', '4,206', '4,258', '4,298', '4,404', '4,491', '4,630', '4,817', '4,843', '5,189', '5,402', '5,495', '5,602', '5,670', '5,691', '5,708', '6,009', '6,081', '6,187', '6,812', '6,972', '6,999', '7,291', '7,438', '854', '906']


In [240]:
from urllib.request import urlopen
from urllib import parse
from bs4 import BeautifulSoup
import json
import time

#page_base = "https://ko.aliexpress.com/w/wholesale-%EC%97%AC%EC%84%B1-%ED%99%94%EC%9E%A5%ED%92%88-%EB%A7%A4%EC%89%AC-%ED%8C%8C%EC%9A%B0%EC%B9%98.html?SearchText=%EC%97%AC%EC%84%B1+%ED%99%94%EC%9E%A5%ED%92%88+%EB%A7%A4%EC%89%AC+%ED%8C%8C%EC%9A%B0%EC%B9%98&catId=0&g=y&initiative_id=SB_20230620200505&spm=a2g0o.productlist.1000002.0&trafficChannel=main"

keyword = '여성 화장품 매쉬 파우치'
page_base = f"https://ko.aliexpress.com/w/wholesale-{parse.quote(keyword.replace(' ', '-'))}.html?SearchText={parse.quote(keyword.replace(' ', '+'))}"

def get_url(page_number):
   if page_number == 1:
      return page_base
   return page_base + "&page=" + str(page_number)

def get_soap(page_number, url):
   #print(f'[soup]')
   #print('    [url]', page_number, url)
   with urlopen(url) as response:
      soup = BeautifulSoup(response, 'html.parser')
   return soup

def get_jscript(soup):
   #print('    [jscript...]')
   jscript = ''
   jcount = 0
   for item in soup.find_all('script'):
      if item.text.find('hierarchy') == -1:
         continue
      jcount += 1
      #print('        jscript', jcount)
      jscript = item.text
      break
   return jscript

def get_json(jscript):
   #print('    [json...]')
   begin_text = '"root":["'
   end_text = '"'
   begin = jscript.find(begin_text)
   if begin == -1:
      return None
   end = jscript.find(end_text, begin + len(begin_text))
   if end == -1:
      return None

   begin_text = '"data"'
   end_text = '"storeDirect_3641"'
   begin = jscript.find(begin_text, end)
   if begin == -1:
      return None
   end = jscript.find(end_text, begin + len(begin_text))
   if end == -1:
      return None

   text = '{' + jscript[begin:end].strip().strip(',') + '}}'
   return json.loads(text)

dic_productId = {}

def do_parsing():
   #print('    [parsing...]')
   num = 0
   if 'itemList' not in data_json['data']['root']['fields']['mods']:
      #print(data_json['data']['root']['fields']['mods'])
      return False
   for content in data_json['data']['root']['fields']['mods']['itemList']['content']:
      num += 1
      #print(content['trace']['utLogMap']['formatted_price'])
      
      displayTitle = content['title']['displayTitle']
      productId = content['productId']
      
      if productId not in dic_productId:
         dic_productId[productId] = 1
      else:
         continue
      if 'originalPrice' in content['prices']:
         price = int(content['prices']['originalPrice']['formattedPrice'].replace('₩ ', '').replace(',', ''))
      else:
         price = int(content['prices']['salePrice']['formattedPrice'].replace('₩ ', '').replace(',', ''))
      if price < 1000:
         print(num, price, displayTitle[:10], f'https://ko.aliexpress.com/item/{productId}.html')
   return True

for page_number in range(1, 11):
   time.sleep(1)
   stop_retry = False
   for retry in range(1, 10):
      if stop_retry == True:
         break
      url = get_url(page_number)
      soup = get_soap(page_number, url)
      jscript = get_jscript(soup)
      if len(jscript) == 0:
         stop_retry = True
         continue
      data_json = get_json(jscript)
      if data_json is None:
         stop_retry = True
         continue
      succeeded = do_parsing()
      if succeeded == True:
         stop_retry = True
         continue
      else:
         stop_retry = False
         continue
      

2 92 여성용 메이크업 가 https://ko.aliexpress.com/item/1005003202199441.html
3 92 메쉬 화장품 메이크 https://ko.aliexpress.com/item/1005003096220831.html
4 92 투명 블랙 메이크업 https://ko.aliexpress.com/item/1005003709696634.html
9 93 여성용 화장품 가방 https://ko.aliexpress.com/item/1005004938663184.html
11 92 여성용 캐주얼 지퍼 https://ko.aliexpress.com/item/1005003843166538.html
13 612 캐주얼 패션 투명  https://ko.aliexpress.com/item/1005005433491273.html
18 817 메이크업 브러쉬 여 https://ko.aliexpress.com/item/1005004914387474.html
19 92 메이크업 파우치 립 https://ko.aliexpress.com/item/1005004795322341.html
27 93 남녀공용 휴대용 화 https://ko.aliexpress.com/item/1005001814424917.html
37 93 여성용 탈착식 메이 https://ko.aliexpress.com/item/1005004842512461.html
44 92 메쉬 투명 휴대용  https://ko.aliexpress.com/item/1005004509072799.html
42 772 새로운 여성 여행  https://ko.aliexpress.com/item/33012360214.html
44 985 블랙 캐주얼 지퍼  https://ko.aliexpress.com/item/1005004665325472.html
18 865 여행용 블랙 메쉬  https://ko.aliexpress.com/item/1005004351531872.html
29 838 여성용 화장품 가방 https: